In [ ]:
import os

os.environ.setdefault("JAX_PLATFORM_NAME", "cpu")
os.environ.setdefault("JAX_ENABLE_X64", "True")

import numpy as np
from scipy.optimize import root

import jax.numpy as jnp

from docomo_bt_management.dsl import (
    D,
    DiscretizationRequest,
    discretization,
)

In [ ]:
def f(x, y, t):
    return D(x, "t", order=2) + y + t


grid = jnp.linspace(0.0, 1.0, 5, dtype=jnp.float64)
f_tilde, restore_blocks = discretization(
    f,
    DiscretizationRequest(grid=grid, scheme="central"),
)

z = jnp.concatenate(
    [
        jnp.linspace(-0.25, 1.25, 7, dtype=jnp.float64),
        jnp.linspace(10.0, 14.0, 5, dtype=jnp.float64),
    ]
)
blocks = restore_blocks(z)

assert restore_blocks.names() == ("x", "y")
assert blocks["x"].shape == (7,)
assert blocks["y"].shape == (5,)

actual = f_tilde(z)
expected = jnp.array([10.0, 11.25, 12.5, 13.75, 15.0], dtype=jnp.float64)
assert actual.shape == (5,)
assert bool(jnp.allclose(actual, expected, atol=1e-4))
actual

In [ ]:
def second_order_polynomial(x, t):
    return D(x, "t", order=2) + x


second_grid = jnp.linspace(0.0, 1.0, 5, dtype=jnp.float64)
second_tilde, _ = discretization(
    second_order_polynomial,
    DiscretizationRequest(grid=second_grid, scheme="central"),
)
second_halo_points = jnp.linspace(-0.25, 1.25, 7, dtype=jnp.float64)
second_actual = second_tilde(second_halo_points**2)
second_expected = 2.0 + second_grid**2
assert bool(jnp.allclose(second_actual, second_expected, atol=1e-4))


def fourth_order_polynomial(x, t):
    return D(x, "t", order=4) + x


fourth_tilde, _ = discretization(
    fourth_order_polynomial,
    DiscretizationRequest(grid=grid, scheme="central"),
)
fourth_halo_points = jnp.linspace(-0.5, 1.5, 9, dtype=jnp.float64)
fourth_actual = fourth_tilde(fourth_halo_points**4)
fourth_expected = 24.0 + grid**4
assert bool(jnp.allclose(fourth_actual, fourth_expected, atol=1e-3))
second_actual, fourth_actual

In [ ]:
def nonlinear_residual(x, t):
    exact = 1.0 + t**2
    return D(x, "t", order=2) + x**2 - (2.0 + exact**2)


nonlinear_grid = jnp.linspace(0.0, 1.0, 6, dtype=jnp.float64)
halo_points = jnp.linspace(-0.2, 1.2, 8, dtype=jnp.float64)
exact_z = 1.0 + halo_points**2
nonlinear_tilde, nonlinear_blocks = discretization(
    nonlinear_residual,
    DiscretizationRequest(grid=nonlinear_grid, scheme="central"),
)


def nonlinear_system(z_values):
    z_array = jnp.asarray(z_values, dtype=jnp.float64)
    ode_residual = np.asarray(nonlinear_tilde(z_array), dtype=float)
    boundary_residual = np.asarray(
        jnp.array([z_array[0] - exact_z[0], z_array[-1] - exact_z[-1]]),
        dtype=float,
    )
    return np.concatenate([ode_residual, boundary_residual])


initial = np.asarray(
    exact_z + 0.05 * jnp.sin(jnp.arange(exact_z.size)),
    dtype=float,
)
solution = root(nonlinear_system, initial)
assert solution.success
assert np.linalg.norm(nonlinear_system(solution.x), ord=np.inf) < 5e-5

solved_block = nonlinear_blocks(jnp.asarray(solution.x, dtype=jnp.float64))["x"]
solved_grid_values = solved_block[1:-1]
exact_grid_values = 1.0 + nonlinear_grid**2
assert bool(jnp.allclose(solved_grid_values, exact_grid_values, atol=2e-3))
solution.success, np.linalg.norm(nonlinear_system(solution.x), ord=np.inf), solved_grid_values

In [ ]:
def path(t):
    return t**3


d2_path = D(path, "t", order=2)
assert bool(jnp.allclose(d2_path(2.0), 12.0))

In [ ]:
for bad_z in (z[:-1], jnp.concatenate([z, jnp.array([0.0], dtype=z.dtype)])):
    try:
        f_tilde(bad_z)
    except TypeError:
        pass
    else:
        raise AssertionError("bad z length did not fail")


for bad_function in (lambda x: x, lambda x, t, y: x + y + t):
    try:
        discretization(bad_function, DiscretizationRequest(grid=grid))
    except ValueError:
        pass
    else:
        raise AssertionError("bad t signature did not fail")